In [ ]:
import os

os.chdir("./mcp-net/")

In [ ]:
!git clone --depth 1 https://github.com/SLDGroup/G-CASCADE.git 2>&1 | tail -5

In [ ]:
!python acdc_baselines/prepare_acdc_for_gcascade.py

Split loaded: 90 train / 5 dev / 5 test patients
Wrote 1694 slices to /content/drive/MyDrive/Colab_Notebooks/mcp-net/data/ACDC_gcascade/train, list: /content/drive/MyDrive/Colab_Notebooks/mcp-net/data/ACDC_gcascade/lists_ACDC/train.txt
Wrote 104 slices to /content/drive/MyDrive/Colab_Notebooks/mcp-net/data/ACDC_gcascade/valid, list: /content/drive/MyDrive/Colab_Notebooks/mcp-net/data/ACDC_gcascade/lists_ACDC/valid.txt
Wrote 10 volumes to /content/drive/MyDrive/Colab_Notebooks/mcp-net/data/ACDC_gcascade_test, list: /content/drive/MyDrive/Colab_Notebooks/mcp-net/data/ACDC_gcascade/lists_ACDC/test.txt

Done. Point train_ACDC.py at:
  --root_dir /content/drive/MyDrive/Colab_Notebooks/mcp-net/data/ACDC_gcascade
  --volume_path /content/drive/MyDrive/Colab_Notebooks/mcp-net/data/ACDC_gcascade_test
  --list_dir /content/drive/MyDrive/Colab_Notebooks/mcp-net/data/ACDC_gcascade/lists_ACDC


In [ ]:
!cd 'G-CASCADE' && pip install -r requirements.txt

In [ ]:
!pip install medpy==0.5.2

In [ ]:
!pip install thop==0.1.1

In [ ]:
!pip install timm==0.6.13

In [ ]:
!pip install gdown -q

In [ ]:
!cd 'G-CASCADE' && mkdir -p "./pretrained_pth/pvt"
!cd 'G-CASCADE' && gdown --folder "https://drive.google.com/drive/folders/1d5F1VjEF1AtTkNO93JwVBBSivE8zImiF" -O "./pretrained_pth/pvt"

In [ ]:
!pip install segmentation-mask-overlay==0.3.4

In [ ]:
!cd 'G-CASCADE' && python train_ACDC.py --root_dir ../data/GCASCADE/ACDC_gcascade \
  --volume_path ../data/GCASCADE/ACDC_gcascade_test \
  --list_dir ../data/GCASCADE/ACDC_gcascade/lists_ACDC \
  --seed 5 \
  --deterministic 1

In [ ]:
!cd 'G-CASCADE' && python test_ACDC.py --encoder PVT --skip_aggregation additive \
  --batch_size 12 --lr 0.0001 --max_epochs 400 --img_size 224 \
  --save_path ./model_pth/ACDC --seed 5 \
  --root_dir ./data/ACDC/ \
  --volume_path ../data/GCASCADE/ACDC_gcascade_test \
  --list_dir ../data/GCASCADE/ACDC_gcascade/lists_ACDC \
  --deterministic 1

## Benchmark Gcascade

In [ ]:
import time

import numpy as np
import torch
from torch.utils.flop_counter import FlopCounterMode


def count_params(model):
    return sum(p.numel() for p in model.parameters())


def count_flops(model, input_shape, device):
    dummy = torch.randn(1, *input_shape).to(device)
    model.eval()
    with torch.no_grad(), FlopCounterMode(display=False) as fcm:
        model(dummy)
    return fcm.get_total_flops()


def benchmark_inference_time(model, input_shape, device, n_warmup=10, n_runs=50):
    model.eval()
    dummy = torch.randn(1, *input_shape).to(device)

    with torch.no_grad():
        for _ in range(n_warmup):
            _ = model(dummy)
        if device.type == "cuda":
            torch.cuda.synchronize()

        times = []
        for _ in range(n_runs):
            start = time.perf_counter()
            _ = model(dummy)
            if device.type == "cuda":
                torch.cuda.synchronize()
            times.append(time.perf_counter() - start)

    return np.mean(times), np.std(times)


def benchmark_peak_memory(model, input_shape, device):
    if device.type != "cuda":
        return None  # peak memory tracking only meaningful on GPU here
    model.eval()
    dummy = torch.randn(1, *input_shape).to(device)
    torch.cuda.reset_peak_memory_stats(device)
    with torch.no_grad():
        _ = model(dummy)
    torch.cuda.synchronize()
    return torch.cuda.max_memory_allocated(device) / (1024 ** 2)  # MB


def benchmark_pytorch_model(model, input_shape, model_name, device=None, slices_per_volume=10):
    device = device or torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)

    n_params = count_params(model)
    flops = count_flops(model, input_shape, device)
    mean_time_per_slice, std_time_per_slice = benchmark_inference_time(model, input_shape, device)
    peak_mem_mb = benchmark_peak_memory(model, input_shape, device)

    print(f"=== {model_name} ===")
    print(f"Parameters: {n_params:,}")
    print(f"FLOPs: {flops/1e9:.3f} GFLOPs")
    print(f"Inference time per slice: {mean_time_per_slice*1000:.2f} ± {std_time_per_slice*1000:.2f} ms")
    print(f"Extrapolated time per volume ({slices_per_volume} slices, sequential): "
          f"{mean_time_per_slice*slices_per_volume*1000:.2f} ms")
    if peak_mem_mb is not None:
        print(f"Peak GPU memory: {peak_mem_mb:.1f} MB")
    else:
        print("Peak GPU memory: N/A (running on CPU)")
    print(f"Device: {device}")
    print()

    return {
        "model_name": model_name, "params": n_params, "flops": flops,
        "mean_time_per_slice_ms": mean_time_per_slice * 1000,
        "std_time_per_slice_ms": std_time_per_slice * 1000,
        "time_per_volume_ms": mean_time_per_slice * slices_per_volume * 1000,
        "peak_memory_mb": peak_mem_mb, "device": str(device),
    }


In [ ]:
import os

os.chdir('G-CASCADE')

In [ ]:
import sys
import torch

# run this FROM WITHIN your G-CASCADE repo directory (or add it to sys.path),
# since it needs lib.networks -- same reasoning as nnU-Net/TransUNet needing
# their own environments
# sys.path.insert(0, ".")  # adjust if not running from the G-CASCADE root

from lib.networks import PVT_GCASCADE
# from benchmark_pytorch_efficiency import benchmark_pytorch_model

# ============================== CONFIG ============================== #
ENCODER = "PVT"  # "PVT" or "MERIT" -- match whatever you actually trained with
NUM_CLASSES = 4
IMG_SIZE = 224
SKIP_AGGREGATION = "additive"  # match your training command

CHECKPOINT_PATH = "model_pth/ACDC/PVT_GCASCADE_MUTATION_w3_7_Run1_224/PVT_GCASCADE_MUTATION_w3_7_Run1_pretrain_epo400_bs12_lr0.0001_224_s5/best.pth"  # your actual trained checkpoint
# =================================================================================== #


def build_model():
    if ENCODER == "PVT":
        model = PVT_GCASCADE(n_class=NUM_CLASSES, img_size=IMG_SIZE, k=11, padding=5,
                              conv="mr", gcb_act="gelu", skip_aggregation=SKIP_AGGREGATION)

    else:
        raise ValueError(f"Unknown encoder: {ENCODER}")
    return model


def main():
    model = build_model()

    checkpoint = torch.load(CHECKPOINT_PATH, map_location="cpu")
    # handle both plain state_dict checkpoints (best.pth/epoch_N.pth) and
    # the resume-checkpoint wrapper dict, same distinction that caused the
    # earlier TransUNet loading error -- model weights don't actually
    # matter for a pure efficiency benchmark, but load correctly regardless
    if isinstance(checkpoint, dict) and "model_state_dict" in checkpoint:
        model.load_state_dict(checkpoint["model_state_dict"])
    else:
        model.load_state_dict(checkpoint)

    benchmark_pytorch_model(model, (1, IMG_SIZE, IMG_SIZE), f"G-CASCADE-{ENCODER}")


In [ ]:
main()

using relative_pos
using relative_pos
using relative_pos
using relative_pos
Model GCASCADE decoder:  created, param count: 4315363
=== G-CASCADE-PVT ===
Parameters: 29,169,343
FLOPs: 9.070 GFLOPs
Inference time per slice: 23.08 ± 1.80 ms
Extrapolated time per volume (10 slices, sequential): 230.80 ms
Peak GPU memory: 155.1 MB
Device: cuda



# M&Ms Evaluation

In [ ]:
!python ../mms_baselines/prepare_mms_for_gcascade.py

In [ ]:
!python test_ACDC.py \
  --volume_path ../data/GCASCADE/mms_for_gcascade_test \
  --list_dir ../data/GCASCADE/mms_for_gcascade_test/lists_ACDC \
  --root_dir ./data/ACDC/ \
  --encoder PVT --skip_aggregation additive --batch_size 12 --lr 0.0001 \
  --max_epochs 400 --img_size 224 --save_path ./model_pth/ACDC --seed 5